# Phase 5: Final Results, Confidence Intervals, and Manuscript Tables

This notebook uses the Phase 4 modeling output ZIP to create publication-ready results.

It:

1. Loads repeated cross-validated predictions and cycle-holdout results.
2. Calculates bootstrap 95% confidence intervals for AUROC, AUPRC, and Brier score.
3. Produces testing-budget tables for the best model.
4. Summarizes cycle-holdout transportability.
5. Creates publication-ready figures.
6. Exports manuscript-ready CSV tables and a ZIP.

Upload `Cystatin_C_Phase4_Modeling_Outputs.zip` when prompted.

In [ ]:
# Cell 1 — Imports and folders

import json
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve,
    precision_recall_curve
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

ROOT = Path("/content/cystatin_c_phase5")
INPUT = ROOT / "input"
OUTPUT = ROOT / "outputs"
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_BOOTSTRAP = 2000

print("✅ Environment ready")

In [ ]:
# Cell 2 — Upload and extract Phase 4 ZIP

from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

if len(zip_names) != 1:
    raise ValueError(
        "Upload exactly one ZIP file: "
        "Cystatin_C_Phase4_Modeling_Outputs.zip"
    )

zip_path = INPUT / zip_names[0]
zip_path.write_bytes(uploaded[zip_names[0]])

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(INPUT)

required_files = [
    "analysis_summary.json",
    "repeated_cv_aggregate_metrics.csv",
    "repeated_cv_predictions.csv",
    "testing_budget_results.csv",
    "cycle_holdout_results.csv",
    "best_model_calibration.csv",
    "best_model_permutation_importance.csv",
]

missing = [f for f in required_files if not (INPUT / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing files in ZIP: {missing}")

summary = json.loads((INPUT / "analysis_summary.json").read_text())
metrics = pd.read_csv(INPUT / "repeated_cv_aggregate_metrics.csv")
predictions = pd.read_csv(INPUT / "repeated_cv_predictions.csv")
budgets = pd.read_csv(INPUT / "testing_budget_results.csv")
cycle_results = pd.read_csv(INPUT / "cycle_holdout_results.csv")
calibration = pd.read_csv(INPUT / "best_model_calibration.csv")
importance = pd.read_csv(INPUT / "best_model_permutation_importance.csv")

best_model = summary["best_model"]

print("✅ Results loaded")
print("Best model:", best_model)
print("Participants:", summary["participants"])
print("Cases:", summary["cases"])

In [ ]:
# Cell 3 — Bootstrap confidence intervals

def bootstrap_metric_ci(y, p, metric_function, n_bootstrap=2000, seed=42):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    p = np.asarray(p)
    n = len(y)
    values = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        y_b = y[idx]
        p_b = p[idx]

        # AUROC/AUPRC require both classes.
        if len(np.unique(y_b)) < 2:
            continue

        values.append(metric_function(y_b, p_b))

    values = np.asarray(values)

    return {
        "Estimate": metric_function(y, p),
        "Lower 95% CI": np.percentile(values, 2.5),
        "Upper 95% CI": np.percentile(values, 97.5),
    }


y = predictions["Observed"].astype(int).to_numpy()

ci_rows = []

for model_name in [
    "Logistic regression",
    "Elastic net",
    "Random forest",
    "XGBoost"
]:
    p = predictions[model_name].to_numpy()

    for metric_name, function in [
        ("AUROC", roc_auc_score),
        ("AUPRC", average_precision_score),
        ("Brier score", brier_score_loss),
    ]:
        result = bootstrap_metric_ci(
            y,
            p,
            function,
            n_bootstrap=N_BOOTSTRAP,
            seed=RANDOM_STATE
        )

        ci_rows.append({
            "Model": model_name,
            "Metric": metric_name,
            **result
        })

bootstrap_ci = pd.DataFrame(ci_rows)

display(bootstrap_ci.round(3))

In [ ]:
# Cell 4 — Manuscript-ready model performance table

performance_wide = (
    bootstrap_ci.pivot(
        index="Model",
        columns="Metric",
        values=["Estimate", "Lower 95% CI", "Upper 95% CI"]
    )
)

performance_rows = []

for model_name in bootstrap_ci["Model"].unique():
    subset = bootstrap_ci.loc[bootstrap_ci["Model"] == model_name]

    row = {"Model": model_name}

    for metric_name in ["AUROC", "AUPRC", "Brier score"]:
        metric_row = subset.loc[subset["Metric"] == metric_name].iloc[0]
        row[metric_name] = (
            f"{metric_row['Estimate']:.3f} "
            f"({metric_row['Lower 95% CI']:.3f}–"
            f"{metric_row['Upper 95% CI']:.3f})"
        )

    performance_rows.append(row)

manuscript_performance = pd.DataFrame(performance_rows)
display(manuscript_performance)

In [ ]:
# Cell 5 — Best-model testing-budget table

best_budget = budgets.loc[budgets["Model"] == best_model].copy()

best_budget = best_budget[
    [
        "Testing budget (%)",
        "People tested",
        "Cases detected",
        "Case detection (%)",
        "Positive yield (%)",
        "Number needed to test",
    ]
]

display(best_budget.round(2))

In [ ]:
# Cell 6 — Cycle-holdout summary

best_cycle = cycle_results.loc[
    cycle_results["Model"] == best_model
].copy()

best_cycle = best_cycle[
    [
        "Train cycle",
        "Test cycle",
        "Test N",
        "Test cases",
        "AUROC",
        "AUPRC",
        "Brier",
    ]
]

display(best_cycle.round(3))

In [ ]:
# Cell 7 — ROC figure

plt.figure(figsize=(7, 6))

for model_name in [
    "Logistic regression",
    "Elastic net",
    "Random forest",
    "XGBoost"
]:
    p = predictions[model_name]
    fpr, tpr, _ = roc_curve(y, p)
    auc_value = roc_auc_score(y, p)
    plt.plot(fpr, tpr, label=f"{model_name} ({auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False-positive rate")
plt.ylabel("True-positive rate")
plt.title("Cross-Validated ROC Curves")
plt.legend()
plt.tight_layout()

roc_path = OUTPUT / "Figure_1_ROC.png"
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 8 — Precision-recall figure

plt.figure(figsize=(7, 6))

for model_name in [
    "Logistic regression",
    "Elastic net",
    "Random forest",
    "XGBoost"
]:
    p = predictions[model_name]
    precision, recall, _ = precision_recall_curve(y, p)
    ap = average_precision_score(y, p)
    plt.plot(recall, precision, label=f"{model_name} ({ap:.3f})")

plt.axhline(y.mean(), linestyle="--", label=f"Prevalence ({y.mean():.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Cross-Validated Precision–Recall Curves")
plt.legend()
plt.tight_layout()

pr_path = OUTPUT / "Figure_2_Precision_Recall.png"
plt.savefig(pr_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 9 — Calibration figure

plt.figure(figsize=(6, 6))
plt.plot(
    calibration["Mean_predicted_risk"],
    calibration["Observed_prevalence"],
    marker="o"
)
plt.plot([0, 0.35], [0, 0.35], linestyle="--")
plt.xlim(0, 0.35)
plt.ylim(0, 0.35)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed prevalence")
plt.title(f"Calibration of {best_model}")
plt.tight_layout()

cal_path = OUTPUT / "Figure_3_Calibration.png"
plt.savefig(cal_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 10 — Feature-importance figure

top_importance = importance.head(12).sort_values(
    "Importance mean",
    ascending=True
)

plt.figure(figsize=(8, 6))
plt.barh(
    top_importance["Feature"],
    top_importance["Importance mean"]
)
plt.xlabel("Decrease in AUROC after permutation")
plt.ylabel("Predictor")
plt.title(f"Permutation Importance: {best_model}")
plt.tight_layout()

importance_path = OUTPUT / "Figure_4_Feature_Importance.png"
plt.savefig(importance_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 11 — Save manuscript tables

bootstrap_path = OUTPUT / "Table_Model_Performance_Bootstrap_CI.csv"
manuscript_path = OUTPUT / "Table_Model_Performance_Formatted.csv"
budget_path = OUTPUT / "Table_Testing_Budget.csv"
cycle_path = OUTPUT / "Table_Cycle_Holdout.csv"
importance_csv = OUTPUT / "Table_Feature_Importance.csv"
calibration_csv = OUTPUT / "Table_Calibration_Deciles.csv"

bootstrap_ci.to_csv(bootstrap_path, index=False)
manuscript_performance.to_csv(manuscript_path, index=False)
best_budget.to_csv(budget_path, index=False)
best_cycle.to_csv(cycle_path, index=False)
importance.to_csv(importance_csv, index=False)
calibration.to_csv(calibration_csv, index=False)

print("✅ Manuscript tables saved")

In [ ]:
# Cell 12 — Generate concise results text

best_metrics = metrics.loc[metrics["Model"] == best_model].iloc[0]
top20 = best_budget.loc[
    best_budget["Testing budget (%)"] == 20
].iloc[0]

results_text = f'''
Among {summary["participants"]:,} adults, {summary["cases"]:,}
({100 * summary["prevalence"]:.2f}%) met the primary definition of
30% negative eGFR discordance. {best_model} achieved the highest
cross-validated discrimination, with an AUROC of
{best_metrics["AUROC"]:.3f} and an AUPRC of {best_metrics["AUPRC"]:.3f}.
When cystatin C testing was targeted to the 20% of participants with
the highest predicted risk, the model identified
{int(top20["Cases detected"])} of {summary["cases"]} discordance cases
({top20["Case detection (%)"]:.1f}%), with a positive yield of
{top20["Positive yield (%)"]:.1f}% and a number needed to test of
{top20["Number needed to test"]:.2f}.
'''.strip()

print(results_text)

(OUTPUT / "Draft_Results_Paragraph.txt").write_text(results_text)

In [ ]:
# Cell 13 — Download final results ZIP

output_zip = Path("/content/Cystatin_C_Phase5_Final_Results.zip")

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT.iterdir():
        zf.write(path, arcname=path.name)

print("✅ Created:", output_zip)
files.download(str(output_zip))

## Upload next

Upload `Cystatin_C_Phase5_Final_Results.zip`.

After that, the study is ready for manuscript drafting, with one remaining methodological note: the 0.50 probability threshold should not be emphasized because the model is intended to rank patients for targeted testing rather than serve as a binary diagnosis at that threshold.